## TRAINING DATA

In [2]:
import os
import cv2
import numpy as np
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt

In [9]:
# Define your dataset paths
TRAIN_DIR = r'C:\Users\igles\OneDrive\Documents\MOR-TEAM-13\TRAINING FOLDER\Banana picker.v5i.folder\train\Lakatan'      # Should contain ONLY normal images
VALID_DIR = r'C:\Users\igles\OneDrive\Documents\MOR-TEAM-13\TRAINING FOLDER\Banana picker.v5i.folder\valid\Lakatan'      # Should contain mixed normal and anomaly
TEST_DIR  = r'C:\Users\igles\OneDrive\Documents\MOR-TEAM-13\TRAINING FOLDER\Banana picker.v5i.folder\test\Lakatan'       # Should contain mixed normal and anomaly

# Image settings
IMG_SIZE = (64, 64)  # Resize images to this size to reduce processing time

In [10]:
def load_data(folder, is_train=True):
    images = []
    labels = []
    
    # If it's training, we assume all data is normal (Label: 1)
    # If it's valid/test, we expect subfolders e.g., folder/normal/ and folder/anomaly/
    
    if is_train:
        # Load all files in the train directory
        for filename in os.listdir(folder):
            img_path = os.path.join(folder, filename)
            try:
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE) # Read as grayscale
                if img is not None:
                    img = cv2.resize(img, IMG_SIZE)
                    images.append(img.flatten())
                    labels.append(1) # 1 represents "Normal" (Inlier)
            except Exception as e:
                continue
    else:
        # Load from subfolders (e.g., 'normal' and 'anomaly')
        for label_name in os.listdir(folder):
            class_folder = os.path.join(folder, label_name)
            if not os.path.isdir(class_folder):
                continue
                
            # Assign label: 1 for 'normal', -1 for anything else (e.g., 'anomaly')
            label = 1 if label_name.lower() == 'normal' else -1
            
            for filename in os.listdir(class_folder):
                img_path = os.path.join(class_folder, filename)
                try:
                    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                    if img is not None:
                        img = cv2.resize(img, IMG_SIZE)
                        images.append(img.flatten())
                        labels.append(label)
                except Exception as e:
                    continue

    return np.array(images), np.array(labels)

In [13]:
print("Loading training data...")
X_train, _ = load_data(TRAIN_DIR, is_train=True)
print(f"Train Shape: {X_train.shape}")

Loading training data...
Train Shape: (280, 4096)


In [ ]:
print("Loading validation data...")
X_valid, y_valid = load_data(VALID_DIR, is_train=False)

print(f"Valid Shape: {X_valid.shape}")

Loading validation data...
Valid Shape: (0,)


In [7]:
# Normalize pixel values (0-255 -> 0.0-1.0)
X_train = X_train / 255.0
X_valid = X_valid / 255.0

In [8]:
# Initialize the model
# nu: Approximate percentage of outliers you expect in the data
model = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05) 

print("Training model...")
model.fit(X_train)
print("Training complete.")

Training model...


ValueError: Expected 2D array, got 1D array instead:
array=[].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [ ]:
print("Predicting on Validation set...")
y_pred_valid = model.predict(X_valid)

# Since the model outputs 1 (Normal) and -1 (Anomaly), 
# we can compare it directly with y_valid if we used the same labeling convention in Step 3.

print("\nValidation Results:")
print(f"Accuracy: {accuracy_score(y_valid, y_pred_valid):.4f}")
print("\nClassification Report:")
# target_names maps 1 to 'Normal' and -1 to 'Anomaly' for the report
print(classification_report(y_valid, y_pred_valid, target_names=['Anomaly', 'Normal']))

In [ ]:
def plot_predictions(X, y_true, y_pred, num_images=5):
    plt.figure(figsize=(15, 5))
    for i in range(num_images):
        # Reshape back to image size for plotting
        img = X[i].reshape(IMG_SIZE)
        
        # Determine titles
        true_label = "Normal" if y_true[i] == 1 else "Anomaly"
        pred_label = "Normal" if y_pred[i] == 1 else "Anomaly"
        color = "green" if y_true[i] == y_pred[i] else "red"
        
        plt.subplot(1, num_images, i+1)
        plt.imshow(img, cmap='gray')
        plt.title(f"True: {true_label}\nPred: {pred_label}", color=color)
        plt.axis('off')
    plt.show()

# Plot first 5 images from validation set
plot_predictions(X_valid, y_valid, y_pred_valid)